# Brute Force Approach for Minimum fuel Trajectories in Earth Moon System

This notebook applies a brute force approach to solve the problem of launching a rocket from Low Earth Orbit (LEO) to Low Moon Orbit (LMO).

Two impulsive burns are appied. One at LEO and one at LMO.

The time of flight and phase of departure are also optimized

### Imports

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from cr3bp import (
    create_earth_moon_system,
    grid_search_method,
    shrink_ranges,
    file_path
)

### Initialize the System / Problem

In [2]:
# Create the Earth-Moon system using the cr3bp module
em = create_earth_moon_system()
print(em.info())

CR3BP System Information:
  Primary 1 mass: 5.972e+24 kg
  Primary 2 mass: 7.342e+22 kg
  Primary 1 radius: 6.371e+06 m
  Primary 2 radius: 1.737e+06 m
  Total mass: 6.045e+24 kg
  Distance: 3.844e+08 m (384400.0 km)
  Mass parameter μ: 0.012145

Characteristic scales:
  Length (l*): 3.844e+08 m (384400.0 km)
  Time (t*): 3.752e+05 s (4.343 days)
  Velocity (v*): 1.025e+03 m/s (1.025 km/s)
  Acceleration (a*): 2.731e-03 m/s^2
  Period: 27.285 days
None


In [3]:
# Define the LEO and LMO altitudes in meters
leo_alt_m=463e3
lmo_alt_m=100e3

In [4]:
# 10km in natural units
print(f"10km in natural units = {10e3/em.l_star}")

10km in natural units = 2.6014568158168575e-05


## Run the Optimization Method

In [5]:
grid_size = (20, 20, 20, 20)  # (num_theta, num_delta_v, num_delta_v_angle, num_tof)

### Iteration 1 - Coarse Grid Search

In [14]:
dec_var_ranges = [[3.9, 4.0], [3.0, 3.1], [0.08, 0.15], [0.68, 0.80]]

In [ ]:
optimals_iteration1 = Path(f"{file_path}/optimals_iteration1.npy")

if optimals_iteration1.exists():
    optimals_iteration1 = np.load(optimals_iteration1, allow_pickle=True)
    print("Loaded existing results")
    print(optimals_iteration1)
else:
    results_df = grid_search_method(em, grid_size, dec_var_ranges, 2.6e-5, leo_alt_m, lmo_alt_m)
    optimals_iteration1 = results_df.iloc[0][['theta', 'delta_v', 'delta_v_angle', 'tof']].values
    np.save(f"{file_path}/optimals_iteration1.npy", optimals_iteration1)
    np.save(f"{file_path}/optimals_iteration1_df.npy", results_df)   
    print("Ran grid search and saved")
    print(optimals_iteration1)
    print(results_df)

Performing grid search with 20 x 20 x 20 x 20 = 160000 grid points...


In [9]:
results_df_iteration1 = np.load(f"{file_path}/optimals_iteration1_df.npy", allow_pickle=True)
print(results_df_iteration1[0:5])

[[ 3.94210526e+00  3.03157895e+00  9.42105263e-02  7.02631579e-01
   4.48571756e+00  1.26346335e-05]
 [ 3.95789474e+00  3.03157895e+00  1.20000000e-01  7.50000000e-01
   4.55757646e+00  8.12410029e-06]
 [ 3.93684211e+00  3.03157895e+00  7.57894737e-02  6.81578947e-01
   4.84997767e+00 -1.61146470e-05]
 [ 3.90526316e+00  3.03684211e+00  1.20000000e-01  6.92105263e-01
   6.59580410e+00 -6.86092919e-06]
 [ 3.95263158e+00  3.02105263e+00  5.36842105e-02  7.07894737e-01
   6.86108766e+00 -5.68311187e-06]]


In [ ]:
optimals_iteration1 = results_df_iteration1[0:5]

0.75


### Iteration 2

In [ ]:
optimals_iteration2 = Path(f"{file_path}/optimals_iteration2.npy")

if optimals_iteration2.exists():
    dec_var_ranges2 = shrink_ranges(optimals_iteration1, dec_var_ranges, shrink_factor=0.5)
    optimals_iteration2_df = np.load(f"{file_path}/optimals_iteration2_df.npy", allow_pickle=True)
    print(f"New dec_var_ranges2: {dec_var_ranges2}")
    optimals_iteration2 = np.load(optimals_iteration2, allow_pickle=True)
    print("Loaded existing results")
    print(optimals_iteration2)
else:
    dec_var_ranges2 = shrink_ranges(optimals_iteration1, dec_var_ranges, shrink_factor=0.5)
    print(f"New dec_var_ranges2: {dec_var_ranges2}")
    optimals_iteration2_df = grid_search_method(em, grid_size, dec_var_ranges2, 9e-6, leo_alt_m, lmo_alt_m)
    optimals_iteration2 = optimals_iteration2_df.iloc[0][['theta', 'delta_v', 'delta_v_angle', 'tof']].values
    np.save(f"{file_path}/optimals_iteration2.npy", optimals_iteration2)
    np.save(f"{file_path}/optimals_iteration2_df.npy", optimals_iteration2_df)   
    print("Ran grid search and saved")
    print(optimals_iteration2)
    print(optimals_iteration2_df)

New dec_var_ranges2: [[np.float64(3.917105263157895), np.float64(3.9671052631578947)], [np.float64(3.0065789473684212), np.float64(3.056578947368421)], [np.float64(0.07671052631578947), np.float64(0.11171052631578947)], [np.float64(0.6776315789473684), np.float64(0.7276315789473684)]]
Performing grid search with 20 x 20 x 20 x 20 = 160000 grid points...
Iteration: 3703/160000
Grid point satisfies constraint: Theta=3.92 rad, Delta_v=3.03, Delta_v_angle=0.09 rad, TOF=0.68 s => Distance to LMO=-2.69 km, Total Delta_v=7.01 km/s
New optimal found: Delta_v=3.10 km/s, Theta=3.92 rad, Delta_v_angle=0.09 rad, TOF=0.68 s, Distance to LMO=-2.69 km
Iteration: 4371/160000
Grid point satisfies constraint: Theta=3.92 rad, Delta_v=3.03, Delta_v_angle=0.11 rad, TOF=0.70 s => Distance to LMO=2.35 km, Total Delta_v=6.74 km/s
New optimal found: Delta_v=3.11 km/s, Theta=3.92 rad, Delta_v_angle=0.11 rad, TOF=0.70 s, Distance to LMO=2.35 km
Error evaluating grid point: Theta=3.92 rad, Delta_v=3.04, Delta_v_a

### Iteration 3

In [ ]:
optimals_iteration3 = Path(f"{file_path}/optimals_iteration3.npy")

if optimals_iteration3.exists():
    optimals_iteration3 = np.load(optimals_iteration3, allow_pickle=True)
    optimals_iteration3_df = np.load(f"{file_path}/optimals_iteration3_df.npy", allow_pickle=True)
    print("Loaded existing results")
    print(optimals_iteration3)
else:
    dec_var_ranges3 = shrink_ranges(optimals_iteration2, dec_var_ranges2, shrink_factor=0.5)
    print(f"New dec_var_ranges3: {dec_var_ranges3}")
    print(f"Optimal iteration 2: {optimals_iteration2}")
    optimals_iteration3_df = grid_search_method(em, grid_size, dec_var_ranges3, 3e-6, leo_alt_m, lmo_alt_m)
    optimals_iteration3 = optimals_iteration3_df.iloc[0][['theta', 'delta_v', 'delta_v_angle', 'tof']].values
    np.save(f"{file_path}/optimals_iteration3.npy", optimals_iteration3)
    np.save(f"{file_path}/optimals_iteration3_df.npy", optimals_iteration3_df)   
    print("Ran grid search and saved")
    print(optimals_iteration3)
    print(optimals_iteration3_df)

Loaded existing results
[3.96961565 3.02484418 0.07460938 0.72992717]


In [ ]:
optimals_iteration3_df = pd.DataFrame(optimals_iteration3_df,columns=['theta', 'delta_v', 'delta_v_angle', 'tof', 'total_delta_v','distance_to_llo'])